![](../img/logo_ucm.jpg)

# Serialización de modelos

En los anteriores Notebooks hemos seguido ejemplos no realistas donde el modelo se entrenaba y exponía directamente sin experimentación, sin guardado, sin trazabilidad... En el siguiente Notebook vamos a trabajar en las distintas opciones para construir un artefacto software que sea guardado para su posterior consumo. 

La serialización de modelos es el proceso de convertir un modelo entrenado en un formato que se puede **guardar en un archivo**. Esto permite que el modelo pueda ser deserializado posteriormente y utilizado para hacer predicciones.

## Pasos a seguir

1. Primero, se entrena un modelo de ML utilizando un conjunto de datos de entrenamiento.

2. Una vez que el modelo ha sido entrenado, se utiliza una librería para convertir el modelo en una secuencia de bytes y guardarlo en un archivo. Librerías comunes para esta tarea incluyen **pickle** y **joblib** en Python

3. Cuando se necesita usar el modelo para hacer predicciones, se lee el archivo y se deserializa el modelo, reconstruyendo el objeto original del modelo entrenado.

Veamos un ejemplo via `pickle`.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import seaborn as sns
import pandas as pd
import numpy as np

iris = sns.load_dataset("iris")
iris.sample(5, random_state=99)

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X = iris[['sepal_length','sepal_width','petal_length','petal_width']]
y = iris['species']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=99)

model = DecisionTreeClassifier(random_state=99)
model.fit(X_train, y_train)
model.score(X_test, y_test)

0.9333333333333333

In [2]:
import pickle

# Utilizamos la librería pickle para guardar el modelo a disco. Aprovechamos para añadir un versionado a dicho modelo.
with open('decisiontree-v01.pkl', 'wb') as file:
    pickle.dump(model, file)

Al ejecutar el código anterior, descubriremos en nuestra carpeta un fichero con el nombre `decisiontree-v01.pkl`. El formato es solo lectura por lo que necesitaremos la librería `pickle` para deserializarlo.

Tratemos ahora de crear una variable nueva y cargar este modelo en dicha variable para comprobar que es completamente funcional.

In [3]:
with open('decisiontree-v01.pkl', 'rb') as file:
    model2 = pickle.load(file)
    
model2.predict(X_test) # este código debería haber fallado si no fuera un modelo de sklearn

array(['virginica', 'setosa', 'versicolor', 'virginica', 'setosa',
       'versicolor', 'versicolor', 'virginica', 'setosa', 'virginica',
       'virginica', 'versicolor', 'virginica', 'versicolor', 'virginica',
       'setosa', 'setosa', 'versicolor', 'versicolor', 'versicolor',
       'versicolor', 'versicolor', 'setosa', 'versicolor', 'virginica',
       'setosa', 'setosa', 'virginica', 'versicolor', 'virginica'],
      dtype=object)

In [4]:
model2

DecisionTreeClassifier(random_state=99)

## Ventajas de la Serialización de Modelos de ML

- **Portabilidad**: los modelos serializados pueden ser transferidos entre diferentes sistemas y entornos, facilitando la colaboración y el despliegue en producción.

- **Reutilización**: pueden ser reutilizados en diferentes aplicaciones o sistemas, permitiendo que se apliquen los mismos modelos en diferentes contextos sin cambios.

- **Consistencia**: garantiza que el modelo utilizado en producción sea exactamente el mismo que el modelo que se entrenó y evaluó, evitando discrepancias que puedan surgir por cambios en el código de entrenamiento.

## Limitaciones de las serialización de Modelos con `pickle`

1. **Dependencia de la estructura del código**

Si usamos clases personalizadas o funciones definidas localmente, deben existir con la misma ruta y nombre al deserializar o de lo contrario obtendremos un error.  

```python

# custom_model.py

class CustomTransformer:
   def transform(self, X):
      return X * 2

import pickle

transformer = CustomTransformer()

with open("transformer.pkl", "wb") as f:
   pickle.dump(transformer, f)

# Desde otro fichero load_model.py

import pickle

with open("transformer.pkl", "rb") as f:
   transformer = pickle.load(f)
```

```bash
# Obtendríamos el siguiente error: 
AttributeError: Can't get attribute 'CustomTransformer' on <module '__main__'>
```

Para solucionarlo, deberemos importar las clases en el fichero load_model.py

```bash
mi_proyecto/
├── custom_transformer.py
├── save_model.py
└── load_model.py
```

```python

# custom_transformer.py

class CustomTransformer:
   def transform(self, X):
      return X * 2

# save_model.py

import pickle
from custom_transformer import CustomTransformer # Necesario que exista e importe para el guardado

transformer = CustomTransformer()

with open("transformer.pkl", "wb") as f:
   pickle.dump(transformer, f)


# load_model.py

import pickle
from custom_transformer import CustomTransformer  # Necesario que exista e importe

with open("transformer.pkl", "rb") as f:
   transformer = pickle.load(f)

print(transformer.transform(10))  # Output: 20
```

---

2. **Compatibilidad con versiones**

`pickle` no es robusto frente a cambios de versión en:

- Python  
- scikit-learn  
- numpy, pandas, etc.

Cambiar a una versión nueva de `scikit-learn` puede romper la deserialización si en el entorno productivo tenemos otras versiones. Lo que nos lleva a una de sus principales limitaciones, `pickle` **no guarda el entorno**. 


3. **Al no guardar el entorno**

No hay registro de:

- Dependencias (`scikit-learn`, `numpy`, etc.)
- Versión del modelo: hemos realizado un control de versiones arcaico utilizando el nombre del fichero y pudiendo ser sobreescrita por error o de manera maliciosa por un tercero.
- Los parámetros usados en entrenamiento que nos permitan reproducir el entrenamiento o pasar auditorías sobre el modelo.


4. **No guarda metadatos ni firmas (signature)**

No queda registrado:

- Columnas esperadas a la entrada, como vimos en Pydantic.  
- Estructura de las predicciones.  
- Métricas asociadas al modelo, tags y otros metadatos de interés como equipo encargado, permisos...

Esto lo hace menos útil para colaboración, auditoría o producción.


5. **No versiona ni organiza modelos**

No hay manera nativa de:

- Versionar modelos  
- Comparar entre ejecuciones  
- Acciones que permitan promover a *staging* o *producción*  


6. **Seguridad**

`pickle` puede ejecutar código arbitrario al deserializar. 

Nunca debes cargar un `.pkl` de una fuente no confiable. 

```python
import os

class Evil:
   def __reduce__(self):
      return (os.system, ("rm -rf /",))  # Ejecutaría un borrado del sistema si el script tuviera permisos suficientes. Sino los tuviera podríamos garantizarlos o buscar brechas de seguridad. 

# Esto puede ser serializado y luego ejecutado al cargarlo con pickle.load()
```

En la próxima sección, entraremos en detalle sobre lo que es MLflow, que si bien utiliza `pickle` para cargar los modelos, mitiga estos problemas al disponer de Registro centralizado de modelos, versionado, auditoría y signature del modelo.